# 基线推理 — 作者发布的预训练权重 (MSN_weights3.h5)

这份 notebook 只做一件事:**在修复过对齐问题的数据上,跑一遍原作者的预训练权重**,
得到一个能和 `MSN_train_skullfix.ipynb` 训练结果直接对比的基线数字。

## 为什么要单独建这份,而不是改 `MSN_model_inference_demo.ipynb`

原来那份 demo 有三个问题会污染评估结果:

1. **对齐被重新破坏**。`.ply` 文件本身是修好的(共享坐标系),但 demo 的
   `run_inference` 对 partial、评估 cell 又对 GT,**各自独立**调用了
   `normalize_point_cloud`,各算各的质心和最大半径。实测 skull_000:
   GT→input 最近距离中位数被抬高 **1.31 倍**(0.0219 → 0.0287)。
2. **指标是无量纲的**,没有 `scale_mm` 就没法换算成毫米,和训练结果不可比。
3. **只跑了 50 个样本**,而且是全部 100 个里的前 50,不是验证集。

这份 notebook 改成:直接读 `data/cache/*.npz`(已对齐 + 带 `scale_mm`)、
只评估**训练时那 20 颗验证集颅骨**、指标输出毫米。

## ⚠️ 关于权重加载的一个坑(务必知道)

预训练权重**只能加载进 `src/models/msn_demo_arch.py`**(demo 原始架构的逐字抽取),
**不能**加载进 `src/models/msn_skullfix.py` 的 `paper()`。

后者的 docstring 里写着 "paper() can load MSN_weights3.h5",这句话是**错的**,
而且它的失败方式是**静默的**:`load_weights(by_name=True, skip_mismatch=True)`
不报错就返回,但 32 个权重组里只匹配上 3 个(`D-OUT_lin`/`D1-IN`/`D2-IN`),
**约 96% 的网络仍是随机初始化**,你会得到一堆垃圾预测却看不到任何报错。

原因是层的嵌套方式不同,不是形状不同:demo 的 `LBR` 把 Dense+ReLU 包在一个名为
`E-IN_LBR1` 的嵌套 Model 里,checkpoint 存的是 `E-IN_LBR1/E-IN_LBR1_lin/kernel`;
重写版直接建一个叫 `E-IN_LBR1_lin` 的顶层 Dense,找的是 `E-IN_LBR1_lin/kernel`。
数学一样、形状一样,但路径对不上,`by_name` 匹配不了。注意力块同理
(`E-SA1` 一个组 vs `E-SA1_Q`/`_K`/`_V` 三个)。

所以:**跑预训练权重用 `msn_demo_arch`,跑本项目训练出来的模型用 `msn_skullfix`,
两者之间不要试图互相加载。** 下面第 3 节有一个加载是否真的生效的自检。

## 指标可比性

评估用的是 `msn_skullfix.py` 里的 `calc_cd` / `calc_dcd` —— 和训练时**完全同一套函数**,
所以两边数字定义一致。它相对 demo 版本只是把 `(B,N,M,3)` 的距离张量换成了
`|a|²-2a·b+|b|²`(省约 10 倍显存),数值约定不变(欧氏距离,非平方距离)。

## 1. 配置

In [1]:
import os, sys, json

# 仓库根目录：从当前工作目录向上找，直到看见 src/models/。
# （别写死 "../.."：这个 notebook 在 notebooks/ 下，而 notebooks/demo/ 下的那几个
#   比它深一级，同一个相对路径在两处含义不同，写死很容易错。）
REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
assert os.path.isdir(os.path.join(REPO, "src", "models")), f"没找到仓库根目录，当前在 {os.getcwd()}"
print("REPO =", REPO)
CACHE  = os.path.join(REPO, "data", "cache", "skullfix_pairs_4096_6144.npz")
WEIGHTS= os.path.join(REPO, "msn_downloads", "MSN_weights3.h5")
RUN    = os.path.join(REPO, "experiments", "baseline_es20", "run.json")   # 本项目训练出来的那次
OUT_CSV= os.path.join(REPO, "experiments_log", "pretrained_baseline", "eval_val20.csv")

# 只评估训练时的验证集(公平对比的前提:那 80 颗训练集本项目的模型见过)。
# 想看全部 100 颗就改成 "all"。
WHICH_IDS = "val"      # "val" | "all"

# demo 架构含 BERT,共 296.9M 参数。指标已换成省显存的实现,GPU 通常放得下;
# 若 OOM 就把这里改成 "/CPU:0"(慢很多但一定跑得完)。
DEVICE = "/GPU:0"

os.environ.setdefault("HF_HOME", "/root/.cache/huggingface")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

for p in (CACHE, WEIGHTS, RUN):
    assert os.path.exists(p), f"缺文件: {p}"

sys.path.insert(0, os.path.join(REPO, "src", "models"))
print("配置 OK")

REPO = /root/comp0190-organ-completion
配置 OK


## 2. 数据 — 直接读已对齐的缓存

`.npz` 里的 `inputs`/`gt` 已经在**同一个坐标系**里(由 defective 推出唯一一个相似变换、
同时作用于两者),而且已经采样成 4096 / 6144 点。

**所以这里绝对不要再调 `normalize_point_cloud` 或再做 FPS** —— 那正是 demo 把对齐
重新弄坏的原因。原样喂给模型即可。

In [2]:
import numpy as np

data = np.load(CACHE)
ids, inputs, gt, scale_mm = data["ids"], data["inputs"], data["gt"], data["scale_mm"]
print(f"缓存: {len(ids)} 对 | inputs {inputs.shape} | gt {gt.shape}")

meta = json.load(open(RUN))
val_ids = meta["val_ids"]
print(f"本项目训练那次的划分: train {len(meta['train_ids'])} / val {len(val_ids)} 颗")
print(f"  该次 best val CD_t = {meta['best_val_cd_t_mm']:.2f} mm  (epoch {meta['epochs_run']} 停)")

if WHICH_IDS == "val":
    sel = np.array([i for i, s in enumerate(ids) if s in set(val_ids)])
else:
    sel = np.arange(len(ids))
print(f"\n本次评估 {len(sel)} 颗: {', '.join(ids[sel][:8])}{' ...' if len(sel) > 8 else ''}")

缓存: 100 对 | inputs (100, 4096, 3) | gt (100, 6144, 3)
本项目训练那次的划分: train 80 / val 20 颗
  该次 best val CD_t = 7.08 mm  (epoch 133 停)

本次评估 20 颗: 000, 004, 010, 012, 018, 022, 030, 031 ...


## 3. 建模型 + 加载预训练权重(含加载自检)

用 `msn_demo_arch`(demo 原始架构逐字抽取)。加载用**严格模式** —— 不传 `by_name`、
不传 `skip_mismatch`,这样一旦拓扑对不上会直接抛异常,而不是静默地只加载一部分。

下面还额外做了一次"权重是否真的变了"的自检,因为静默失败是这里最容易踩的坑。

In [3]:
import tensorflow as tf
for _g in tf.config.experimental.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)

import msn_demo_arch as demo

AE = demo.PCT_AE_Multimodal(bert_model=demo.bert_model,
                            PCT_encoder=demo.PCT_encoder,
                            pct_decoder=demo.pct_decoder)
print(f"模型参数: {AE.model.count_params()/1e6:.1f}M (含 BERT)")

# 自检:记下加载前的权重,加载后确认确实被改写
_before = [w.numpy().copy() for w in AE.model.weights[:40]]
AE.model.load_weights(WEIGHTS)          # 严格模式
_changed = sum(1 for b, w in zip(_before, AE.model.weights) if not np.array_equal(b, w.numpy()))
print(f"权重加载自检: 前 40 个张量中有 {_changed} 个被改写")
assert _changed > 30, "权重没有真正加载进来 —— 检查是不是用错了架构模块"
print("✓ 预训练权重已正确加载")

2026-08-05 21:35:49.492281: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-05 21:35:49.492302: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-05 21:35:49.493224: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at bert-base-uncased were not used when initializing TFBertModel: ['nsp___cls', 'mlm___cls']
- This IS expected if you are initializing TFBertModel from the check

模型参数: 296.9M (含 BERT)
权重加载自检: 前 40 个张量中有 40 个被改写
✓ 预训练权重已正确加载


## 4. 推理 + 计算指标

- 输入:`.npz` 里已对齐、已采样好的 4096 点 partial,**不做任何额外处理**
- `eye_seed` 固定为 0(demo 推理时本来就是 `tf.zeros`)
- 类别文本 `"skull"` 只 tokenize 一次(BERT 被冻结且只有一个类别,输出是常量)
- 指标用 `msn_skullfix` 的 `calc_cd`/`calc_dcd`,和训练时同一套定义
- 毫米换算用**每个样本各自的** `scale_mm`

> demo 的 `UniformSampler` 用的是有状态随机数,同一输入两次调用结果会有差异
> (训练 notebook 实测差 1.03)。下面固定了全局种子以尽量稳定,但要完全逐位
> 可复现需要改成 stateless 抽样 —— 这属于改动 demo 架构,这里没做。

In [4]:
import pandas as pd
from tqdm.notebook import tqdm
import msn_skullfix as msn                     # 只借它的指标函数,不建它的模型

tokenizer = demo.BertTokenizer.from_pretrained("bert-base-uncased")
enc = tokenizer.encode_plus("skull", add_special_tokens=True, max_length=128,
                            padding="max_length", truncation=True, return_tensors="tf")

tf.keras.utils.set_random_seed(42)

rows = []
with tf.device(DEVICE):
    for i in tqdm(sel, desc="pretrained baseline"):
        x  = tf.convert_to_tensor(inputs[i][None, ...], dtype=tf.float32)   # (1,4096,3) 原样
        y  = tf.convert_to_tensor(gt[i][None, ...],     dtype=tf.float32)   # (1,6144,3) 原样
        es = tf.zeros([1, 1, 1], dtype=tf.float32)

        pred = AE.model([x, es, enc["input_ids"], enc["attention_mask"]], training=False)

        cd_p, cd_t = msn.calc_cd(pred, y)
        dcd        = msn.calc_dcd(pred, y)

        s = float(scale_mm[i])
        rows.append({"id": ids[i],
                     "CD_t_mm": float(cd_t[0]) * s,
                     "CD_p_mm": float(cd_p[0]) * s,
                     "DCD":     float(dcd),
                     "scale_mm": s})

df = pd.DataFrame(rows)
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
df.to_csv(OUT_CSV, index=False)
print(f"\n{len(df)} 颗评估完成 -> {OUT_CSV}")
df.head()

pretrained baseline:   0%|          | 0/20 [00:00<?, ?it/s]


20 颗评估完成 -> /root/comp0190-organ-completion/experiments_log/pretrained_baseline/eval_val20.csv


,id,CD_t_mm,CD_p_mm,DCD,scale_mm
0,000,9.728243,21.918888,1.433165,99.149811
1,004,10.080171,22.378261,1.403341,99.514618
2,010,11.110958,24.612060,1.426880,109.580299
3,012,9.899926,23.039032,1.434189,108.891243
4,018,11.829884,25.495007,1.443587,109.937019


## 5. 结果 — 与本项目训练的模型对比

两边用的是**同一批验证颅骨、同一套指标定义、同一份已对齐的数据**,所以这张表是可比的。

⚠️ 但要注意这个对比的**性质**:`MSN_weights3.h5` 是原作者在他们自己的数据上训练的,
不是在 SkullFix 颅骨上训的。所以正确的表述是"**预训练权重直接应用于颅骨**
vs **在颅骨上从零训练**",而不是"同一任务下两个方法的较量"。写进论文前建议
先查清原作者训练时是否包含颅骨类数据,再定表述。

In [5]:
print("=" * 62)
print(f"{'':22s} {'CD_t (mm)':>12s} {'CD_p (mm)':>12s} {'DCD':>10s}")
print("-" * 62)
print(f"{'预训练基线 (本次)':22s} {df.CD_t_mm.mean():12.2f} {df.CD_p_mm.mean():12.2f} {df.DCD.mean():10.3f}")
print(f"{'  ├ 中位数':22s} {df.CD_t_mm.median():12.2f} {df.CD_p_mm.median():12.2f} {df.DCD.median():10.3f}")
print(f"{'  └ 标准差':22s} {df.CD_t_mm.std():12.2f} {df.CD_p_mm.std():12.2f} {df.DCD.std():10.3f}")
print("-" * 62)
print(f"{'从零训练 (本工作)':22s} {meta['best_val_cd_t_mm']:12.2f} {'-':>12s} {'-':>10s}")
print("=" * 62)
print(f"\n评估集: 同一批 {len(df)} 颗验证颅骨 | 训练那次 seed={meta['seed']}")

                          CD_t (mm)    CD_p (mm)        DCD
--------------------------------------------------------------
预训练基线 (本次)                    10.71        23.62      1.428
  ├ 中位数                       10.80        23.93      1.434
  └ 标准差                        0.91         1.62      0.020
--------------------------------------------------------------
从零训练 (本工作)                     7.08            -          -

评估集: 同一批 20 颗验证颅骨 | 训练那次 seed=42


## 6. 可视化(抽一颗看)

In [6]:
import plotly.graph_objects as go

k = int(sel[0])
with tf.device(DEVICE):
    pred = AE.model([tf.convert_to_tensor(inputs[k][None, ...], dtype=tf.float32),
                     tf.zeros([1, 1, 1]), enc["input_ids"], enc["attention_mask"]],
                    training=False).numpy()[0]

fig = go.Figure()
for pts, name, color, size in [(inputs[k], "input (partial)", "#D32F2F", 1.6),
                               (pred,      "pretrained pred", "#1565C0", 1.6),
                               (gt[k],     "ground truth",    "#2E7D32", 1.2)]:
    fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers",
                               marker=dict(size=size, color=color, opacity=0.55), name=name))
fig.update_layout(title=f"skull_{ids[k]} — 预训练权重", scene=dict(aspectmode="data"),
                  margin=dict(l=0, r=0, b=0, t=40), height=650)
fig.show()